# 예제 02. 결측값과 자료형 처리
빅데이터프로그래밍 · 3주차

## 목표
- 결측값을 삭제 · 대체로 처리한다
- `object` 자료형을 숫자로 바꾼다
- 범주형 값을 숫자로 바꾼다

모델은 빈칸과 문자를 받지 못합니다. 이 두 가지를 없애는 것이 전처리의 절반입니다.


In [ ]:
csv = """name,gender,department,study_hours,attendance,midterm,final
김통계,남,통계학과,12.5,95%,88,92
이확률,여,통계학과,8.0,88%,92,85
박회귀,남,컴퓨터공학과,,72%,79,68
최추정,여,통계학과,15.0,100%,95,98
정검정,남,경제학과,4.5,61%,61,
한분산,여,컴퓨터공학과,10.0,90%,84,88
오평균,남,경제학과,6.5,,70,74
서표본,여,통계학과,13.0,97%,100,96
남표준,남,컴퓨터공학과,9.5,85%,66,71
윤편차,여,경제학과,,80%,82,79
"""

with open("students.csv", "w", encoding="utf-8") as f:
    f.write(csv)

print("students.csv 생성 완료")


In [ ]:
import pandas as pd

df = pd.read_csv("students.csv")
print(df.dtypes)
print()
print(df.isnull().sum())


## 1. 결측값 — 어떻게 처리할지 먼저 정합니다

| 방법 | 명령 | 언제 |
| --- | --- | --- |
| 행 삭제 | `dropna()` | 결측이 적고 데이터가 충분할 때 |
| 값 대체 | `fillna(값)` | 데이터가 아까울 때 |
| 그대로 | — | 결측 자체가 정보일 때 (드묾) |


In [ ]:
# 방법 A — 결측이 있는 행 삭제
dropped = df.dropna()
print("원래:", len(df), "→ 삭제 후:", len(dropped))


In [ ]:
# 방법 B — 평균으로 대체 (수치 열)
filled = df.copy()
filled["study_hours"] = filled["study_hours"].fillna(filled["study_hours"].mean())
filled["final"] = filled["final"].fillna(filled["final"].mean())

print(filled[["study_hours", "final"]].round(2))
print("\n남은 결측:", filled.isnull().sum().sum())


## 2. object 자료형 — 숫자처럼 보이지만 문자입니다
`attendance` 는 `95%` 형태의 문자열입니다. 그대로는 계산되지 않습니다.


In [ ]:
print(df["attendance"].dtype)
print(df["attendance"].head())

# 계산을 시도하면 실패합니다
try:
    df["attendance"].mean()
except TypeError as err:
    print("TypeError:", err)


In [ ]:
# % 를 떼고 숫자로 바꿉니다
filled["attendance"] = (filled["attendance"]
                        .str.replace("%", "", regex=False)
                        .astype(float))

print(filled["attendance"].dtype)
print(filled["attendance"].head())
print("평균 출석률:", filled["attendance"].mean())


In [ ]:
# 남은 결측(출석률)도 채웁니다
filled["attendance"] = filled["attendance"].fillna(filled["attendance"].median())
print(filled.isnull().sum().sum(), "개 결측 남음")


## 3. 범주형 → 숫자
- 값이 두 개면 0/1 로 바꾸는 것이 간단합니다 (`map`)
- 값이 셋 이상이면 원-핫 인코딩을 씁니다 (`get_dummies`)


In [ ]:
filled["gender"] = filled["gender"].map({"남": 0, "여": 1})
print(filled["gender"].head())


In [ ]:
encoded = pd.get_dummies(filled, columns=["department"], dtype=int)
print(encoded.columns.tolist())
encoded.head(3)


## 4. 왜 숫자 하나로 바꾸지 않는가
학과를 0·1·2 로 바꾸면 "경제학과의 두 배가 통계학과" 라는 잘못된 크기 관계가 생깁니다.
순서가 없는 범주는 원-핫으로 각각 독립된 열로 만듭니다.


## 5. 처리 전후 비교


In [ ]:
before = pd.read_csv("students.csv")

summary = pd.DataFrame({
    "before_dtype": before.dtypes.astype(str),
    "before_null": before.isnull().sum(),
})
print(summary)
print("\n처리 후 결측:", encoded.isnull().sum().sum())
print("처리 후 object 열:", (encoded.dtypes == "object").sum())


## 직접 해보기
1. `final` 의 결측을 평균이 아니라 `midterm` 값으로 채워 보세요.
2. `gender` 를 `get_dummies` 로 바꾸면 열이 몇 개가 되는지 확인하세요.


In [ ]:
# 여기에 작성하세요
